# Improved Solution

The starter notebook provides a Logistic Regression baseline.

To improve F1 score, we:
- Added feature engineering
- Included categorical variables
- Used One-Hot Encoding
- Trained an XGBoost classifier
- Tuned the classification threshold using public_test.csv

In [1]:
import pandas as pd

print(pd.__version__)

2.0.3


In [2]:
import pandas as pd
import numpy as np

In [3]:
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

print("Train:", train.shape)
print("Public:", public_test.shape)
print("Private:", private_test.shape)

Train: (10000, 14)
Public: (3000, 14)
Private: (3000, 13)


In [4]:
train.head()

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             10000 non-null  int64  
 1   Age                 8520 non-null   float64
 2   Income              9016 non-null   float64
 3   City_Tier           10000 non-null  int64  
 4   Device_Type         10000 non-null  object 
 5   Traffic_Source      10000 non-null  object 
 6   Pages_Viewed        10000 non-null  int64  
 7   Products_Viewed     10000 non-null  int64  
 8   Time_On_Site        8152 non-null   float64
 9   Previous_Purchases  10000 non-null  int64  
 10  Discount_Seen       10000 non-null  int64  
 11  Browser_Version     10000 non-null  int64  
 12  Campaign_Code       10000 non-null  int64  
 13  Converted           10000 non-null  int64  
dtypes: float64(3), int64(9), object(2)
memory usage: 1.1+ MB


In [6]:
train.isnull().sum()

User_ID                  0
Age                   1480
Income                 984
City_Tier                0
Device_Type              0
Traffic_Source           0
Pages_Viewed             0
Products_Viewed          0
Time_On_Site          1848
Previous_Purchases       0
Discount_Seen            0
Browser_Version          0
Campaign_Code            0
Converted                0
dtype: int64

In [7]:
train["Converted"].value_counts()

Converted
0    6913
1    3087
Name: count, dtype: int64

In [8]:
train.shape

(10000, 14)

In [9]:
def create_features(df):

    df = df.copy()

    df["Pages_per_Product"] = (
        df["Pages_Viewed"] /
        (df["Products_Viewed"] + 1)
    )

    df["Income_per_Age"] = (
        df["Income"] /
        (df["Age"] + 1)
    )

    df["Engagement"] = (
        df["Pages_Viewed"] *
        df["Time_On_Site"]
    )

    return df

In [10]:
train = create_features(train)
public_test = create_features(public_test)
private_test = create_features(private_test)

In [11]:
TARGET = "Converted"

X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]

X_public = public_test.drop(columns=[TARGET])
y_public = public_test[TARGET]

In [12]:
categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical:")
print(categorical_cols)

print("\nNumerical:")
print(numerical_cols)

Categorical:
['Device_Type', 'Traffic_Source']

Numerical:
['User_ID', 'Age', 'Income', 'City_Tier', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases', 'Discount_Seen', 'Browser_Version', 'Campaign_Code', 'Pages_per_Product', 'Income_per_Age', 'Engagement']


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import f1_score

from xgboost import XGBClassifier

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numerical_cols
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_cols
        )
    ]
)

In [16]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

In [17]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [18]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['User_ID', 'Age', 'Income',
                                                   'City_Tier', 'Pages_Viewed',
                                                   'Products_Viewed',
                                                   'Time_On_Site',
                                                   'Previous_Purchases',
                                                   'Discount_Seen',
                                                   'Browser_Version',
                                                   'Campaign_Code',
                                                   'Pages_per_Product',
                                                   'Income_per_Age',
                                                   'Engagement']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   Sim...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

In [19]:
probs = pipeline.predict_proba(X_public)[:,1]

In [20]:
import numpy as np

best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.10, 0.90, 0.01):

    preds = (probs >= threshold).astype(int)

    score = f1_score(y_public, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

Best Threshold: 0.2699999999999999
Best F1: 0.5391014975041597


In [21]:
combined = pd.concat(
    [train, public_test],
    ignore_index=True
)

X_final = combined.drop(columns=["Converted"])
y_final = combined["Converted"]

pipeline.fit(X_final, y_final)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['User_ID', 'Age', 'Income',
                                                   'City_Tier', 'Pages_Viewed',
                                                   'Products_Viewed',
                                                   'Time_On_Site',
                                                   'Previous_Purchases',
                                                   'Discount_Seen',
                                                   'Browser_Version',
                                                   'Campaign_Code',
                                                   'Pages_per_Product',
                                                   'Income_per_Age',
                                                   'Engagement']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   Sim...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

In [22]:
private_probs = pipeline.predict_proba(
    private_test
)[:,1]

private_preds = (
    private_probs >= best_threshold
).astype(int)

In [23]:
submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": private_preds
})

submission.head()

,User_ID,Converted
0,103001,0
1,103002,0
2,103003,1
3,103004,1
4,103005,0


In [24]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created!")

submission.csv created!


# Summer Analytics Week 2 Hackathon
## Starter Notebook

This notebook walks you through a simple end-to-end pipeline to get your first submission on the leaderboard. It covers:

- Loading the provided datasets
- Basic preprocessing (handling missing values & encoding)
- Training a **Logistic Regression** baseline model
- Generating predictions for `private_test.csv`
- Creating a valid `submission.csv`

> **Note:** This is just a starting point. Feel free to experiment and improve upon this baseline!


### Step 1: Import Libraries

We start by importing `pandas` for loading and manipulating data, and a few utilities from `scikit-learn` for preprocessing and modelling.
These are all standard libraries that come pre-installed in most Python environments like Kaggle, Colab, or Anaconda.
No additional installations are required to run this notebook.


In [14]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


### Step 2: Load Data

We read all three CSV files — `train.csv`, `public_test.csv`, and `private_test.csv` — into pandas DataFrames.
The training set contains labelled examples we will learn from, while the private test set is what we need to make final predictions on.
Printing the shapes gives a quick sanity check that the files loaded correctly.


In [ ]:
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test.shape)

train.head()


### Step 3: Define Features and Target

We separate the target column (`Converted`) from the rest of the features in the training set.
For simplicity, we only keep **numeric columns** in this baseline — categorical columns are dropped for now.
This keeps things easy to understand before we add more advanced preprocessing.


In [ ]:
TARGET = "Converted"

# Separate features and target
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]

# Keep only numeric columns for this baseline
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
X_train = X_train[numeric_cols]
X_private = private_test[numeric_cols]

print("Features used:", numeric_cols)


### Step 4: Handle Missing Values & Scale Features

Real-world datasets often have missing values — we fill them in using the **median** of each column, which is robust to outliers.
After that, we apply **Standard Scaling** to bring all features onto the same scale, which is important for Logistic Regression to work well.
We fit both the imputer and scaler only on the training data, then apply the same transformation to the test data to avoid data leakage.


In [ ]:
# Fill missing values with the median of each column
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_private_imputed = imputer.transform(X_private)

# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_private_scaled = scaler.transform(X_private_imputed)

print("Preprocessing done! Training data shape:", X_train_scaled.shape)


### Step 5: Train a Logistic Regression Model

Logistic Regression is one of the simplest and most interpretable classification algorithms — a great starting point for any binary classification problem.
It models the probability that a given input belongs to a class using a linear combination of features passed through a sigmoid function.
We set `max_iter=1000` to ensure the solver has enough iterations to converge on this dataset.


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print("Training Accuracy: {:.4f}".format(model.score(X_train_scaled, y_train)))


### Step 6: Generate Predictions and Create Submission

We run the trained model on the scaled private test data to generate our final predictions.
The predictions are combined with the `User_ID` column into a DataFrame matching the required submission format.
The file is saved as `submission.csv` — just upload this to the competition portal to appear on the leaderboard!


In [ ]:
predictions = model.predict(X_private_scaled)

submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
submission.head()
